# 심화 미션: 스마트팜 온실 출하 기록
- 상황: 선별대에서 도장을 찍기 전에 등외를 미리 알고 싶다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 온실에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 상품 / 등외 | 선별대에서 붙이는 판정. 등외는 제값에 못 파는 것 |
| 양액 (EC) | 물에 녹인 거름. 그 진하기를 dS/m 라는 단위로 잰다 |
| 산도 (pH) | 산성인지 알칼리성인지. 7이 중간이고 낮을수록 산성 |
| 토양 수분 | 흙에 물기가 얼마나 있는지 (%) |
| 야간 최저 기온 | 밤에 가장 낮았던 기온. 작물이 스트레스를 받는 지점 |

## Q1. 파일 열고 크기 확인하기

In [12]:
import pandas as pd

df = pd.read_csv("../../data/day03_greenhouse.csv")

print("행 수, 열 수:", df.shape)
df.head()

행 수, 열 수: (2000, 13)


,batch_id,harvested_at,house_id,crop,temp_avg,humidity_avg,co2_ppm,soil_moisture,ec,ph,light_hours,night_temp_min,result
0,B-0001,2026-03-01 06:00,A동,딸기,21.7,79.0,727,55.5,1.42,6.33,6.5,12.6,상품
1,B-0002,2026-03-01 07:12,B동,파프리카,23.4,65.3,694,63.5,2.28,5.99,4.8,13.0,상품
2,B-0003,2026-03-01 08:24,C동,파프리카,22.9,65.5,589,58.3,1.54,6.18,6.5,11.8,상품
3,B-0004,2026-03-01 09:36,A동,파프리카,24.1,63.6,546,78.8,1.63,6.31,10.5,12.6,상품
4,B-0005,2026-03-01 10:48,B동,파프리카,22.4,72.0,750,51.1,1.90,6.18,7.3,16.5,상품


## Q2. 등외는 얼마나 드문가

In [13]:
print(df["result"].value_counts())
print(df["result"].value_counts(normalize=True) * 100)

result
상품    1861
등외     139
Name: count, dtype: int64
result
상품    93.05
등외     6.95
Name: proportion, dtype: float64


 ## Q3. 정답표를 숫자로 바꾸기

In [14]:
# result가 "등외"면 1, 아니면 0인 숫자 열을 새로 만든다
df["등외여부"] = (df["result"] == "등외").astype(int)

print(df[["result", "등외여부"]].head())


  result  등외여부
0     상품     0
1     상품     0
2     상품     0
3     상품     0
4     상품     0


In [15]:
print(df["등외여부"].value_counts())

등외여부
0    1861
1     139
Name: count, dtype: int64


## Q4. 입력과 정답으로 가르기

In [16]:
feature_cols = [
    "temp_avg", "humidity_avg", "co2_ppm", "soil_moisture",
    "ec", "ph", "light_hours", "night_temp_min",
]

X = df[feature_cols]
y = df["등외여부"]

print("입력:", X.shape)
print("정답:", y.shape)


입력: (2000, 8)
정답: (2000,)


## Q5. 학습용과 시험용으로 나누기

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("학습용:", len(X_train), "| 등외", (y_train == 1).sum(), round((y_train == 1).mean() * 100, 2), "%")
print("시험용:", len(X_test), "| 등외", (y_test == 1).sum(), round((y_test == 1).mean() * 100, 2), "%")


학습용: 1600 | 등외 111 6.94 %
시험용: 400 | 등외 28 7.0 %


## Q6. 아무것도 배우지 않은 기준 모델

In [18]:
import numpy as np

기준예측 = np.zeros(len(y_test), dtype=int)
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")


기준 모델 정확도: 93.0 %


## Q7. 진짜 모델 학습시키기

In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 단위를 맞추기 위한 표준화
scaler = StandardScaler()
X_train_스케일 = scaler.fit_transform(X_train)
X_test_스케일 = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_스케일, y_train)
예측 = model.predict(X_test_스케일)

print("예측 개수:", len(예측))
print("등외라고 예측한 건수:", (예측 == 1).sum())
print("정확도:", round(accuracy_score(y_test, 예측) * 100, 2), "%")


예측 개수: 400
등외라고 예측한 건수: 20
정확도: 94.5 %


## Q8. 혼동행렬 네 칸 채우기

In [20]:
from sklearn.metrics import confusion_matrix

행렬 = confusion_matrix(y_test, 예측)
print(행렬)

맞힌상품, 헛경보, 놓친등외, 잡은등외 = 행렬.ravel()

print("상품인데 상품이라 함 :", 맞힌상품)
print("상품인데 등외라 함   :", 헛경보, " <- 헛경보")
print("등외인데 상품이라 함 :", 놓친등외, " <- 놓친 등외")
print("등외인데 등외라 함   :", 잡은등외)


[[365   7]
 [ 15  13]]
상품인데 상품이라 함 : 365
상품인데 등외라 함   : 7  <- 헛경보
등외인데 상품이라 함 : 15  <- 놓친 등외
등외인데 등외라 함   : 13


## Q9. 세 가지 지표 구하기

| | 정확도 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|
| 기준 모델 (전부 상품) | | | | |
| 내가 학습시킨 모델 | | | | |

In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_test, 예측, target_names=["상품", "등외"], digits=3))


              precision    recall  f1-score   support

          상품      0.961     0.981     0.971       372
          등외      0.650     0.464     0.542        28

    accuracy                          0.945       400
   macro avg      0.805     0.723     0.756       400
weighted avg      0.939     0.945     0.941       400



## Q10. 놓친 등외를 더 잡으려면

| 문턱 | 잡은 등외 | 헛경보 | 재현율 | 정밀도 |
|---|---|---|---|---|

In [23]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score

# 모델은 다시 학습시키지 않는다. 확률에 문턱만 다르게 적용한다
확률 = model.predict_proba(X_test_스케일)[:, 1]

for 문턱 in [0.5, 0.3, 0.2, 0.1]:
    예측_문턱 = (확률 >= 문턱).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, 예측_문턱).ravel()
    재현율 = recall_score(y_test, 예측_문턱, zero_division=0)
    정밀도 = precision_score(y_test, 예측_문턱, zero_division=0)
    print(f"문턱 {문턱} | 잡은 등외 {tp} | 헛경보 {fp} | 재현율 {round(재현율,3)} | 정밀도 {round(정밀도,3)}")


문턱 0.5 | 잡은 등외 13 | 헛경보 7 | 재현율 0.464 | 정밀도 0.65
문턱 0.3 | 잡은 등외 19 | 헛경보 13 | 재현율 0.679 | 정밀도 0.594
문턱 0.2 | 잡은 등외 21 | 헛경보 19 | 재현율 0.75 | 정밀도 0.525
문턱 0.1 | 잡은 등외 25 | 헛경보 36 | 재현율 0.893 | 정밀도 0.41


## 마무리 - 어디까지 갔나

[심화 미션] 스마트팜 온실 출하 기록
- 어디까지 풀었나 : [Q8까지]
- 막힌 문항 : [5, 10]번
- 왜 막혔나 : [5번은 등외 비율을 양쪽에 맞추는 조건을 어떻게 거는지 몰랐다.
              10번은 문턱을 바꾼다는 게 무슨 뜻인지 감이 안 왔다]
- 오늘 실습과 달랐던 점 : [열이 8개뿐이라 무엇을 넣을지 고민할 일이 없었다.
                        590개일 때는 그 고르는 일이 절반이었다는 걸 알겠다]